In [5]:
import pandas as pd
import math

# 修改成你的 Excel 文件路径
file_path = r"367654152_按分数_虎歌打分_41_41.xlsx"

# 读取 Excel
df = pd.read_excel(file_path)

# 你的表格中，“总分”后面的列都是作品评分
start_col = df.columns.get_loc("总分") + 1
work_cols = df.columns[start_col:]

def calc_final_score(scores):
    # 转成数字，去掉空值，但保留 0 分
    scores = pd.to_numeric(scores, errors="coerce").dropna()

    n = len(scores)
    remove_count = math.ceil(n * 0.10)

    # 从小到大排序
    scores = scores.sort_values()

    # 去掉最低的 remove_count 个，最高的 remove_count 个
    remaining_scores = scores.iloc[remove_count : n - remove_count]

    return pd.Series({
        "打分人数": n,
        "去掉最高分数量": remove_count,
        "去掉最低分数量": remove_count,
        "剩余人数": len(remaining_scores),
        "最终评分": remaining_scores.mean()
    })

In [6]:
# 逐个作品计算
result = df[work_cols].apply(calc_final_score).T

# 整理结果
result = result.reset_index().rename(columns={"index": "作品"})
result["最终评分"] = result["最终评分"].round(4)

# 按最终评分从高到低排序
result = result.sort_values("最终评分", ascending=False).reset_index(drop=True)
result.insert(0, "排名", range(1, len(result) + 1))

# 打印结果
print(result)

    排名                     作品  打分人数  去掉最高分数量  去掉最低分数量  剩余人数    最终评分
0    1                   红中咆哮  41.0      5.0      5.0  31.0  5.0000
1    2                     虎咚  41.0      5.0      5.0  31.0  4.8065
2    3                  混的太逼真  41.0      5.0      5.0  31.0  4.7097
3    4                    孤虎者  41.0      5.0      5.0  31.0  4.2903
4    5                    一挑五  41.0      5.0      5.0  31.0  4.1935
5    6                  红中红红中  41.0      5.0      5.0  31.0  4.1613
6    7                 白莲花的葬礼  41.0      5.0      5.0  31.0  4.1290
7    8               夜访 延迟三毫秒  41.0      5.0      5.0  31.0  4.0323
8    9                     访夜  41.0      5.0      5.0  31.0  4.0323
9   10  进击的巨大！日本小姑娘激情献唱《红中の子》  41.0      5.0      5.0  31.0  3.8387
10  11                    虎阳假  41.0      5.0      5.0  31.0  3.6774
11  12                    话连错  41.0      5.0      5.0  31.0  3.6774
12  13                     群虎  41.0      5.0      5.0  31.0  3.6774
13  14                  讨厌线下梦  41.0      5.0    

In [7]:
# 保存结果
result.to_excel("最终评分结果.xlsx", index=False)